<a href="https://colab.research.google.com/github/jfishovitz/BCCE2026/blob/main/protein_structure_exploration_from_Claude.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Protein Structure Levels & Structural Change
### Primary, Secondary, Tertiary, and Quaternary Structure — and how they respond to post-translational modification (PTM) or ligand binding

**In this notebook you will:**
1. Fetch real PDB structures for a protein in two different states (e.g., with/without a bound ligand, or with/without a PTM)
2. Compare the **primary structure** (sequence)
3. Compare the **secondary structure** content (% helix / % sheet) using DSSP
4. Quantify **tertiary structure** changes (RMSD, domain motions)
5. Quantify **quaternary structure** changes (inter-subunit distances), where relevant
6. Visualize both states in 3D and overlay them directly

**Worked examples included:**
- **Maltose-binding protein (MBP):** ligand-binding-driven tertiary "clamshell" closure (monomeric, no quaternary structure — good for isolating tertiary-level change)
- **Glycogen phosphorylase:** PTM-driven (Ser14 phosphorylation) quaternary structure change (a classic allostery example)

A "plug in your own PDB pair" template is included at the end so you can run this same analysis on a protein of your own choosing.


## 1. Setup
Run this cell first. It installs the packages we need: **Biopython** (structure parsing, sequence/RMSD tools) and **py3Dmol** (interactive 3D visualization).

In [ ]:
# Install dependencies (Colab-specific)
!pip install biopython py3Dmol -q

import Bio
from Bio.PDB import PDBList, PDBParser, Superimposer, DSSP
from Bio.PDB.Polypeptide import three_to_index, index_to_one
import py3Dmol
import numpy as np
import warnings
warnings.filterwarnings("ignore")

print(f"Biopython version: {Bio.__version__}")


## 2. Helper functions

These functions do the underlying work (fetching structures, extracting sequences, extracting CA atoms, computing centroids, and rendering 3D views). You won't need to edit this cell — just run it once. The analysis cells further down call these functions.

In [ ]:
pdbl = PDBList()
parser = PDBParser(QUIET=True)

def fetch_structure(pdb_id, pdir="structures"):
    """Download a PDB file and parse it into a Biopython Structure object."""
    path = pdbl.retrieve_pdb_file(pdb_id, pdir=pdir, file_format="pdb")
    structure = parser.get_structure(pdb_id, path)
    return structure, path

def get_sequence(structure, chain_id="A"):
    """Extract the amino acid sequence (one-letter code) for a given chain."""
    seq = ""
    for residue in structure[0][chain_id]:
        if residue.id[0] == " ":  # skip heteroatoms/waters
            try:
                seq += index_to_one(three_to_index(residue.resname))
            except KeyError:
                seq += "X"
    return seq

def get_ca_atoms(structure, chain_id, start=None, end=None):
    """Return a list of CA (alpha-carbon) atoms for a chain, optionally restricted
    to a residue number range [start, end]."""
    atoms = []
    for res in structure[0][chain_id]:
        if "CA" in res:
            resnum = res.id[1]
            if start is None or (start <= resnum <= end):
                atoms.append(res["CA"])
    return atoms

def chain_centroid(structure, chain_id):
    """Geometric centroid of a chain's CA atoms — a simple proxy for 'where the
    subunit is' when measuring quaternary structure changes."""
    coords = [atom.coord for atom in get_ca_atoms(structure, chain_id)]
    return np.mean(coords, axis=0)

def run_dssp(structure, pdb_file, chain_id="A"):
    """Run DSSP to assign secondary structure. Returns a list of per-residue
    secondary structure codes (H = alpha helix, E = beta strand, etc.)."""
    model = structure[0]
    dssp = DSSP(model, pdb_file, dssp="mkdssp")
    return [dssp[key][2] for key in dssp.keys() if key[0] == chain_id]

def show_structure(pdb_id, width=450, height=450):
    """Cartoon view of a structure fetched directly from the PDB, with any bound
    ligand/heteroatoms shown as sticks."""
    view = py3Dmol.view(query=f"pdb:{pdb_id}", width=width, height=height)
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.addStyle({"hetflag": True}, {"stick": {"colorscheme": "yellowCarbon"}})
    view.zoomTo()
    return view

def show_overlay(path1, path2, color1="blue", color2="orange", width=600, height=500):
    """Overlay two structures (already downloaded PDB files) in a single view."""
    view = py3Dmol.view(width=width, height=height)
    view.addModel(open(path1).read(), "pdb")
    view.addModel(open(path2).read(), "pdb")
    view.setStyle({"model": 0}, {"cartoon": {"color": color1}})
    view.setStyle({"model": 1}, {"cartoon": {"color": color2}})
    view.zoomTo()
    return view

# DSSP binary (needed for secondary structure assignment)
!apt-get install -y dssp -q > /dev/null
print("Helper functions loaded.")


---
## 3. Case Study 1 — Maltose-Binding Protein (MBP): ligand-binding-driven tertiary structure change

MBP is a single-chain (monomeric) periplasmic protein that binds maltose. It's a classic example of a **hinge-bending / "Venus flytrap" mechanism**: two structural lobes close around the sugar when it binds. Because MBP is monomeric, this example isolates the **tertiary structure** level without any quaternary complications.

- **Apo (no ligand):** PDB [`1OMP`](https://www.rcsb.org/structure/1OMP)
- **Holo (maltose-bound):** PDB [`1ANF`](https://www.rcsb.org/structure/1ANF)


In [ ]:
mbp_ids = {"apo": "1OMP", "holo": "1ANF"}  # holo = maltose-bound

mbp_structures = {}
mbp_paths = {}
for label, pdb_id in mbp_ids.items():
    struct, path = fetch_structure(pdb_id)
    mbp_structures[label] = struct
    mbp_paths[label] = path

print("Loaded:", mbp_ids)


### 3a. Primary structure (sequence)
Since apo and holo are the *same protein*, the sequence should be identical. This is a good moment to pause and ask: **if the primary structure is unchanged, where does the functional change come from?**

In [ ]:
for label, struct in mbp_structures.items():
    seq = get_sequence(struct, "A")
    print(f"{label.upper()} ({mbp_ids[label]}), {len(seq)} aa:")
    print(seq[:80] + "...\n")

seq_apo = get_sequence(mbp_structures["apo"], "A")
seq_holo = get_sequence(mbp_structures["holo"], "A")
print("Sequences identical:", seq_apo == seq_holo)


### 3b. Secondary structure (DSSP)
Helix/sheet content should be roughly *unchanged* — the individual secondary structure elements stay intact. What moves is how they're **arranged relative to each other** (tertiary structure).

In [ ]:
for label, struct in mbp_structures.items():
    ss = run_dssp(struct, mbp_paths[label])
    helix = ss.count("H") / len(ss) * 100
    sheet = ss.count("E") / len(ss) * 100
    print(f"{label}: {helix:.1f}% helix, {sheet:.1f}% sheet, n={len(ss)} residues")


### 3c. Tertiary structure: the hinge motion

A whole-chain superposition will show a **large** RMSD — not because the protein unfolded, but because of the hinge rotation between lobes. To confirm it's a rigid-body hinge motion (not local unfolding), we then superimpose each lobe **individually**, which should align much better on its own.

> **Note:** the domain boundary residue numbers below (N-lobe ≈ 1–109, C-lobe ≈ 265–370) are approximate. Before trusting them, confirm/refine against a reliable source (e.g., UniProt domain annotations) or by inspecting the structure visually in the 3D view below — don't take these numbers on faith.

In [ ]:
sup = Superimposer()

def compare_rmsd(struct1, struct2, chain_id, start=None, end=None, label=""):
    fixed = get_ca_atoms(struct1, chain_id, start, end)
    moving = get_ca_atoms(struct2, chain_id, start, end)
    n = min(len(fixed), len(moving))
    sup.set_atoms(fixed[:n], moving[:n])
    print(f"{label} CA-RMSD: {sup.rms:.2f} Å  (n={n} atoms)")
    return sup.rms

print("--- MBP apo vs holo ---")
compare_rmsd(mbp_structures["apo"], mbp_structures["holo"], "A", label="Whole chain")
compare_rmsd(mbp_structures["apo"], mbp_structures["holo"], "A", 1, 109, label="N-lobe only (approx. res 1-109)")
compare_rmsd(mbp_structures["apo"], mbp_structures["holo"], "A", 265, 370, label="C-lobe only (approx. res 265-370)")


### 3d. Quantifying the clamshell closure
Distance between the two lobe centroids — this should be *smaller* in the maltose-bound (holo) form, i.e., the clamshell closing around the ligand.

In [ ]:
def domain_centroid(structure, chain_id, start, end):
    atoms = get_ca_atoms(structure, chain_id, start, end)
    return np.mean([a.coord for a in atoms], axis=0)

for label, struct in mbp_structures.items():
    n_lobe = domain_centroid(struct, "A", 1, 109)
    c_lobe = domain_centroid(struct, "A", 265, 370)
    dist = np.linalg.norm(n_lobe - c_lobe)
    print(f"{label}: N-lobe to C-lobe centroid distance = {dist:.1f} Å")


### 3e. 3D visualization
The bound maltose is shown as yellow sticks in the holo structure.

In [ ]:
print("Apo MBP (no maltose):")
show_structure("1OMP").show()


In [ ]:
print("Holo MBP (maltose-bound — note the closed clamshell around the yellow ligand):")
show_structure("1ANF").show()


### 3f. Overlay: see the hinge motion directly
Blue = apo, orange = holo. Look for the rigid rotation of one lobe relative to the other.

In [ ]:
show_overlay(mbp_paths["apo"], mbp_paths["holo"], color1="blue", color2="orange").show()


**Discussion questions:**
1. Which level of structure changed the most between the apo and holo forms — and which barely changed at all?
2. Why might a whole-chain RMSD be a *misleading* summary statistic for this kind of conformational change?
3. How would you expect this hinge closure to relate to MBP's biological function (transporting maltose into the cell)?


---
## 4. Case Study 2 — Glycogen Phosphorylase: PTM-driven quaternary structure change

This example contrasts with MBP: phosphorylase is a **homodimer**, and a single phosphorylation event (Ser14) on each subunit shifts the whole dimer from a T-state (less active) to an R-state (more active) — illustrating how a small covalent change can propagate to reorganize a **subunit interface**.

- **Phosphorylase b (unphosphorylated, T-state):** PDB [`8GPB`](https://www.rcsb.org/structure/8GPB)
- **Phosphorylase a (phosphorylated at Ser14, R-state):** PDB [`1GPA`](https://www.rcsb.org/structure/1GPA)


In [ ]:
phos_ids = {"b_unphos": "8GPB", "a_phos": "1GPA"}

phos_structures = {}
phos_paths = {}
for label, pdb_id in phos_ids.items():
    struct, path = fetch_structure(pdb_id)
    phos_structures[label] = struct
    phos_paths[label] = path

for label, struct in phos_structures.items():
    chains = [c.id for c in struct[0]]
    print(f"{label} ({phos_ids[label]}): chains present = {chains}")


### 4a. Secondary & tertiary structure near the phosphorylation site
Reuse the same helper functions from Case Study 1. Local structure near residue 14 should barely change — the effect is felt further away, at the dimer interface.

In [ ]:
for label, struct in phos_structures.items():
    ss = run_dssp(struct, phos_paths[label], chain_id="A")
    helix = ss.count("H") / len(ss) * 100
    sheet = ss.count("E") / len(ss) * 100
    print(f"{label}: {helix:.1f}% helix, {sheet:.1f}% sheet")

print()
compare_rmsd(phos_structures["b_unphos"], phos_structures["a_phos"], "A", label="Chain A whole-chain")


### 4b. Quaternary structure: dimer interface distance
This is the level where the PTM's effect is most visible — measure the distance between the two subunit centroids in each state.

In [ ]:
for label, struct in phos_structures.items():
    chain_ids = [c.id for c in struct[0]]
    if "A" in chain_ids and "B" in chain_ids:
        cA = chain_centroid(struct, "A")
        cB = chain_centroid(struct, "B")
        dist = np.linalg.norm(cA - cB)
        print(f"{label}: chain A-B centroid distance = {dist:.1f} Å")
    else:
        print(f"{label}: only one chain present in this file ({chain_ids}) — "
              f"check the PDB entry, some depositions only contain the asymmetric unit.")


### 4c. Visualize the phosphorylation site (Ser14 / phospho-Ser14)

In [ ]:
def show_site(pdb_id, resnum=14, chain_id="A", width=450, height=450):
    view = py3Dmol.view(query=f"pdb:{pdb_id}", width=width, height=height)
    view.setStyle({"cartoon": {"color": "lightgray"}})
    view.addStyle({"chain": chain_id, "resi": str(resnum)}, {"stick": {"colorscheme": "magentaCarbon"}})
    view.zoomTo({"chain": chain_id, "resi": str(resnum)})
    return view

print("Ser14 region, unphosphorylated (8GPB):")
show_site("8GPB").show()


In [ ]:
print("Ser14 (phosphorylated) region (1GPA):")
show_site("1GPA").show()


**Discussion questions:**
1. Contrast this example with MBP: which structural level was most affected by ligand binding (MBP) vs. by the PTM (phosphorylase)?
2. Phosphorylation adds a single, small chemical group. Why might such a small local change be able to produce a large-scale structural effect elsewhere in the protein? (Keyword: **allostery**.)
3. Can you think of a reason evolution might favor "small covalent switch → large conformational change" as a regulatory strategy, compared to synthesizing/degrading a whole new protein?


---
## 5. Your turn: analyze your own PDB pair

Use this section to run the same style of analysis on **any protein pair you choose** — e.g., an apo/holo enzyme, a different PTM pair, or a conformational-change pair from the literature. Just edit the values in the cell below and re-run the cells that follow it.

**Before you start:** look up your chosen PDB entries on the [RCSB PDB website](https://www.rcsb.org/) and check:
- What chain ID(s) are present (don't assume it's `"A"`)
- Whether the protein is a monomer or a multi-chain complex (relevant for whether quaternary analysis applies)
- Approximate domain boundaries, if you want to do the per-domain RMSD analysis (optional)


In [ ]:
# ============================================
# EDIT THIS CELL: define your comparison here
# ============================================
STATE_1_LABEL = "unmodified"     # e.g. "apo", "unphosphorylated", "deoxy"
STATE_2_LABEL = "modified"       # e.g. "holo", "phosphorylated", "oxy"
PDB_ID_1 = "1OMP"                # <-- replace with your PDB ID
PDB_ID_2 = "1ANF"                # <-- replace with your PDB ID
CHAIN_ID = "A"                   # chain to analyze - check your structure first!

# Optional: only fill these in if you want the tertiary per-domain RMSD analysis
DOMAIN_1_RANGE = None   # e.g. (1, 109), or None to skip
DOMAIN_2_RANGE = None   # e.g. (265, 370), or None to skip
# ============================================

my_ids = {STATE_1_LABEL: PDB_ID_1, STATE_2_LABEL: PDB_ID_2}
my_structures = {}
my_paths = {}
for label, pdb_id in my_ids.items():
    struct, path = fetch_structure(pdb_id)
    my_structures[label] = struct
    my_paths[label] = path

print(f"Loaded {STATE_1_LABEL} ({PDB_ID_1}) and {STATE_2_LABEL} ({PDB_ID_2})")
for label, struct in my_structures.items():
    chains = [c.id for c in struct[0]]
    print(f"  {label}: chains = {chains}")


In [ ]:
# Primary structure: sequence identity check
seq1 = get_sequence(my_structures[STATE_1_LABEL], CHAIN_ID)
seq2 = get_sequence(my_structures[STATE_2_LABEL], CHAIN_ID)
print(f"{STATE_1_LABEL} length: {len(seq1)} aa")
print(f"{STATE_2_LABEL} length: {len(seq2)} aa")
print("Sequences identical:", seq1 == seq2)
if seq1 != seq2:
    print("Note: length/sequence differences often reflect missing loops/termini in the")
    print("crystal structure rather than a true biological sequence difference - check the PDB entry.")


In [ ]:
# Secondary structure content
for label, struct in my_structures.items():
    ss = run_dssp(struct, my_paths[label], chain_id=CHAIN_ID)
    helix = ss.count("H") / len(ss) * 100
    sheet = ss.count("E") / len(ss) * 100
    print(f"{label}: {helix:.1f}% helix, {sheet:.1f}% sheet, n={len(ss)} residues")


In [ ]:
# Tertiary structure: whole-chain and (optional) per-domain RMSD
compare_rmsd(my_structures[STATE_1_LABEL], my_structures[STATE_2_LABEL], CHAIN_ID,
             label=f"Whole chain ({STATE_1_LABEL} vs {STATE_2_LABEL})")

if DOMAIN_1_RANGE and DOMAIN_2_RANGE:
    for name, (start, end) in [("Domain 1", DOMAIN_1_RANGE), ("Domain 2", DOMAIN_2_RANGE)]:
        compare_rmsd(my_structures[STATE_1_LABEL], my_structures[STATE_2_LABEL], CHAIN_ID,
                     start, end, label=f"{name} ({start}-{end})")


In [ ]:
# Quaternary structure: only meaningful if there is more than one chain
chains_present = [c.id for c in my_structures[STATE_1_LABEL][0]]
if len(chains_present) > 1:
    for label, struct in my_structures.items():
        chain_ids = [c.id for c in struct[0]]
        print(f"{label} subunit centroid distances:")
        for i, c1 in enumerate(chain_ids):
            for c2 in chain_ids[i+1:]:
                try:
                    d = np.linalg.norm(chain_centroid(struct, c1) - chain_centroid(struct, c2))
                    print(f"  {c1}-{c2}: {d:.1f} Å")
                except KeyError:
                    continue
else:
    print("Only one chain detected - this protein doesn't have quaternary structure to compare "
          "(or the biological assembly isn't included in this PDB file).")


In [ ]:
# 3D visualization
print(f"{STATE_1_LABEL.upper()} ({PDB_ID_1}):")
show_structure(PDB_ID_1).show()


In [ ]:
print(f"{STATE_2_LABEL.upper()} ({PDB_ID_2}):")
show_structure(PDB_ID_2).show()


In [ ]:
# Overlay both states
show_overlay(my_paths[STATE_1_LABEL], my_paths[STATE_2_LABEL]).show()


**Write-up prompt:** for your chosen pair, summarize in a few sentences:
- Which level(s) of protein structure changed the most?
- Which level(s) stayed essentially constant?
- How does the structural change you observed relate to the protein's function?
